In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.6 MB/s eta 0:00:00


In [3]:
import pandas as pd

# reemplaza por el nombre real
df = pd.read_csv("cuidados_plantas.csv")

df.head()

,id,species_id,name,scientific_name,sunlight,watering,pruning
0,1,352,Ambrosia Apple,"[""Malus 'Ambrosia'""]",Ambrosia Apple (Malus 'Ambrosia') needs at lea...,Ambrosia Apple (Malus 'Ambrosia') plants requi...,The Ambrosia Apple (Malus 'Ambrosia') should b...
1,4,367,Honeycrisp Apple,"[""Malus 'Honeycrisp'""]","Sunlight is essential for all types of apples,...",Honeycrisp Apple (Malus 'Honeycrisp') should b...,Pruning Honeycrisp apples is an essential main...
2,5,1893,mandarin orange,"[""Citrus reticulata 'Clementine'""]",Mandarin orange plants should be pruned annual...,"Generally, mandarin orange trees need about on...",Mandarin oranges need plenty of sunlight to gr...
3,6,1895,navel orange,"[""Citrus sinensis 'Trovita'""]",Navel oranges need a considerable amount of su...,Water your navel orange tree (Citrus sinensis ...,Because Navel orange (Citrus sinensis 'Trovita...
4,7,1896,navel orange,"[""Citrus sinensis 'Washington'""]",The navel orange tree likes plenty of sunlight...,Navel oranges need to be watered 1–2 times per...,Naval orange trees should be pruned around the...


In [5]:
docs = (
    "Plant: " + df["name"] +
    " (" + df["scientific_name"].astype(str) + "). " +
    "Sunlight: " + df["sunlight"].astype(str) + ". " +
    "Watering: " + df["watering"].astype(str) + ". " +
    "Pruning: " + df["pruning"].astype(str)
).tolist()

docs[:3]

['Plant: Ambrosia Apple (["Malus \'Ambrosia\'"]). Sunlight: Ambrosia Apple (Malus \'Ambrosia\') needs at least 8 hours of sunlight each day for optimal growth and fruiting. For best results, the plant should receive direct sunlight for 4 to 6 hours in the morning and again in the afternoon for 2 to 4 hours. A few hours of indirect sunlight in the middle of the day can also provide the additional light needed to help the plant thrive. Too much direct sunlight (especially in the afternoon) can cause leaf scorching and dehydrate the plant, so it is important to provide light shading if your plant is in an area with direct exposure to the sun.. Watering: Ambrosia Apple (Malus \'Ambrosia\') plants require regular watering, especially during the flowering and fruiting season. Generally, they require 1-2 inches of water per week (including rainfall). If the soil is allowed to become too dry, the plant\'s growth and fruit production can be affected. During the spring and summer months, it is i

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(docs, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Index size:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Index size: 3000


In [14]:
def ask_template(question):
    q_emb = embed_model.encode([question])
    D, I = index.search(np.array(q_emb), 1)

    row = df.iloc[I[0][0]]

    return f"""
🌿 {row['name']}
🔬 {row['scientific_name']}

☀️ Luz:
{row['sunlight']}

💧 Riego:
{row['watering']}

✂️ Poda:
{row['pruning']}
"""

In [19]:
ask_template("How to care of a cockins?")

'\n🌿 cockscomb\n🔬 ["Celosia argentea var. cristata (Cristata Group)"]\n\n☀️ Luz:\nCockscomb (Celosia argentea var. cristata) plants require full sun, meaning at least 6 hours of direct sunlight exposure each day to thrive. To ensure maximum growth and health, they should be planted in an area where they\'ll be exposed to direct sunlight most of the day, with some shade in the hottest parts of the afternoon. In USDA Hardiness Zones 9b to 11, cockscomb plants should be given full sun all day long; for those zones below 9b, partial shade is recommended during the hottest part of the day.\n\n💧 Riego:\nCockscomb (Celosia argentea var. cristata) requires moderate to heavy watering during the growing season. Water your cockscomb plants on a regular basis as soon as the soil starts to dry out. Aim to keep the soil moist, but never soggy or waterlogged. Water your cockscomb plants twice a week during the summer months and once a week when the temperatures begin to cool. Cockscomb plants also ap